# Практика № 5. Вариант 7


In [ ]:
%pip install pandas

##  Загрузка данных


In [1]:
import pandas as pd
import timeit
from statistics import median

columns = ["event_id", "user_id", "event_time", "event_type", "page"]
first = pd.read_json("data/events_records.json").T
first.columns = columns
second = pd.read_json("data/events_lines.json", lines=True, convert_dates=False)
events = pd.concat([first, second], ignore_index=True)

# Задаём обычные исходные типы перед оптимизацией.
events = events.astype({"event_id": "int64", "user_id": "int64",
                        "event_time": "object", "event_type": "object", "page": "object"})
display(events.head())
print("Количество событий:", len(events))

,event_id,user_id,event_time,event_type,page
0,1,1085,2024-01-30T20:37:32,page_view,/product/26
1,2,1006,2024-02-24T22:55:13,add_to_cart,/product/28
2,3,1024,2024-02-14T17:49:43,click,/product/27
3,4,1062,2024-02-16T11:21:32,click,/product/1
4,5,1012,2024-02-08T05:25:21,add_to_cart,/


Количество событий: 4000


## 1 Оптимизация типов
Оптимизировать типы данных в таблице событий (включая category )

In [2]:
optimized = events.copy()
for column in ["event_id", "user_id"]:
    optimized[column] = pd.to_numeric(optimized[column], downcast="unsigned")
optimized["event_time"] = pd.to_datetime(optimized["event_time"])
for column in ["event_type", "page"]:
    optimized[column] = optimized[column].astype("category")

display(pd.DataFrame({"До": events.dtypes, "После": optimized.dtypes}))
before = events.memory_usage(deep=True).sum()
after = optimized.memory_usage(deep=True).sum()
print(f"Память до: {before / 1024:.2f} КБ")
print(f"Память после: {after / 1024:.2f} КБ")
print(f"Экономия памяти: {(1 - after / before) * 100:.2f}%")

,До,После
event_id,int64,uint16
user_id,int64,uint16
event_time,object,datetime64[us]
event_type,object,category
page,object,category


Память до: 493.20 КБ
Память после: 56.02 КБ
Экономия памяти: 88.64%


## 2. Сравнение скорости value_counts()
 Сравнить скорость выполнения value_counts() по event_type до и после оптимизации.

In [3]:
counts_before = events["event_type"].value_counts()
counts_after = optimized["event_type"].value_counts()
assert counts_before.to_dict() == counts_after.to_dict()
display(counts_after)

def measure(table):
    times = timeit.repeat(lambda: table["event_type"].value_counts(),
                          repeat=5, number=300)
    return median(times) / 300 * 1000

time_before = measure(events)
time_after = measure(optimized)
print(f"До оптимизации: {time_before:.4f} мс")
print(f"После оптимизации: {time_after:.4f} мс")

event_type
click          819
purchase       807
page_view      804
scroll         788
add_to_cart    782
Name: count, dtype: int64

До оптимизации: 0.3867 мс
После оптимизации: 0.2170 мс


## 3 Вывод
Сделать выводы о влиянии типов данных на скорость.

In [ ]:
print(f"Память уменьшилась на {(1 - after / before) * 100:.2f}%.")
if time_after < time_before:
    print(f"В этом запуске value_counts() быстрее в {time_before / time_after:.2f} раза.")
else:
    print("В этом запуске ускорения value_counts() не получено.")
print("Частоты событий до и после оптимизации совпадают.")